In [13]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/mnt/lareaulab/reliscu/code")

from parse_gtf import *

In [14]:
psi = pd.read_csv(f"data/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_SJ_PSI.csv", index_col=0)
top_qval_mods_df = pd.read_csv("data/enrichments/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strictFALSE_logFC3_nSignif21_enrichments_abund_corr.csv")

In [15]:
data_source = "yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strictFALSE_logFC3_nSignif21_enrichments"

### Add gene names to PSI data

In [16]:
# Parse GTF attribute column
gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCm39/gencode.vM35.annotation.gtf"
gtf = gtf_parse(gtf_file)
gtf_subset = gtf.loc[gtf['feature'].isin(["gene"])]
attrs_df = gtf_subset["attribute"].apply(extract_attributes).apply(pd.Series)
gtf_parsed = pd.concat([gtf_subset.drop(columns=["attribute"]), attrs_df], axis=1)

In [17]:
# Get PSI and GTF data ready to merge on gene IDs
gtf_parsed['gene_id'] = gtf_parsed['gene_id'].str.split(".").str[0]
psi['gene_id'] = psi.index.str.split("_").str[0]
psi['exon_id'] = psi.index.values

In [18]:
psi_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], psi, on="gene_id", how="right")
psi_anno = psi_anno.set_index("exon_id").rename_axis(None)
psi_anno = psi_anno.drop(columns=["gene_id"])

In [19]:
psi_anno

,gene_name,Sample1,Sample2,Sample3,Sample4,Sample5,Sample6,Sample7,Sample8,Sample9,...,Sample191,Sample192,Sample193,Sample194,Sample195,Sample196,Sample197,Sample198,Sample199,Sample200
ENSMUSG00000033845_ProteinCoding_1,Mrpl15,0.993051,0.993742,0.991429,0.990923,0.992510,0.993995,0.991052,0.993992,0.992033,...,0.991705,0.992281,0.991128,0.991474,0.990411,0.992481,0.992344,0.991485,0.991585,0.989809
ENSMUSG00000025903_ProteinCoding_1,Lypla1,0.998125,0.997347,0.996916,0.998937,0.994748,0.993386,0.996764,0.998224,0.998424,...,0.998408,0.999413,0.997840,0.996734,0.998484,0.998364,0.998624,0.998346,0.998623,0.999360
ENSMUSG00000025903_ProteinCoding_2,Lypla1,0.986045,0.986886,0.982486,0.980021,0.988957,0.981987,0.981869,0.989045,0.983026,...,0.986563,0.983836,0.986367,0.978473,0.981423,0.989912,0.987035,0.986580,0.985086,0.988292
ENSMUSG00000002459_ProteinCoding_1,Rgs20,0.989238,0.989849,0.983940,0.986087,0.983217,0.989082,0.985796,0.986845,0.989053,...,0.990198,0.984464,0.987350,0.986844,0.990998,0.974828,0.983984,0.988425,0.989681,0.987894
ENSMUSG00000033793_ProteinCoding_1,Atp6v1h,0.998637,0.998660,0.998095,0.998607,0.997913,0.998648,0.998156,0.998579,0.998150,...,0.998606,0.998411,0.998287,0.998020,0.998059,0.998389,0.998245,0.998480,0.998517,0.998705
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000068457_ProteinCoding_1,Uty,0.590205,0.566337,0.515085,0.581954,0.460589,0.484352,0.546904,0.511290,0.549000,...,0.573166,0.623269,0.547408,0.507886,0.524025,0.580229,0.556364,0.557309,0.434411,0.536232
ENSMUSG00000068457_ProteinCoding_2,Uty,0.996085,0.983782,0.987740,0.996151,0.990953,0.988979,0.999292,0.982738,0.991676,...,0.990527,0.995422,0.990164,0.992162,0.992000,0.990836,0.985215,0.990067,0.985860,0.996448
ENSMUSG00000068457_NMD_3,Uty,1.000000,1.000000,0.996865,1.000000,1.000000,0.982583,1.000000,1.000000,0.997135,...,1.000000,0.991163,1.000000,1.000000,1.000000,0.998893,1.000000,1.000000,0.998566,0.992037
ENSMUSG00000068457_NMD_4,Uty,0.000000,0.035576,0.017825,0.003177,0.000000,0.009457,0.000000,0.000000,0.014717,...,0.015118,0.041860,0.002329,0.029710,0.014309,0.000000,0.010697,0.002343,0.001467,0.004587


### Calc. corr between ME and exon PSI

In [20]:
# Remove modules with < X corr to cell type abundance

mask = ~(top_qval_mods_df['Old_cor'] < .8)
top_qval_mods_df = top_qval_mods_df.loc[mask, :]

In [21]:
corr_df = pd.DataFrame(
    columns=["Gene"] + top_qval_mods_df['Cell_type'].tolist(), 
    index=psi_anno.index
)
corr_df['Gene'] = psi_anno['gene_name'] 

for i, row in top_qval_mods_df.iterrows():
    ctype = row['Cell_type']

    mod_df = pd.read_csv(row['Old_ME_path'])
    mod_eig = mod_df.set_index("Sample")[row['Old_module']]
    mod_eig = pd.to_numeric(mod_eig, errors="coerce")
    
    corrs = psi_anno.iloc[:, 1:].corrwith(mod_eig, axis=1)
    corr_df[ctype] = corrs

In [22]:
corr_df.head()

,Gene,Pvalb,L2_3_IT_CTX,L6b_CTX,Lamp5,L5_PT_CTX,Sncg,L5_6_NP_CTX,Sst_Chodl,Meis2,Oligo,Car3,VLMC,SMC-Peri,Astro,Endo,Micro-PVM
ENSMUSG00000033845_ProteinCoding_1,Mrpl15,-0.069454,-0.004840,0.069912,0.118351,0.122422,0.017886,0.100222,0.026872,-0.008390,-0.092558,0.074651,0.082103,0.056161,-0.111665,-0.149090,-0.130149
ENSMUSG00000025903_ProteinCoding_1,Lypla1,0.076317,0.039982,0.167854,-0.102684,0.076942,0.071249,-0.056183,0.165090,0.033219,-0.048791,0.178308,-0.041318,0.055663,0.066788,-0.131338,-0.512185
ENSMUSG00000025903_ProteinCoding_2,Lypla1,0.097284,-0.030244,0.158590,0.071056,0.048305,-0.024337,-0.025743,0.114916,0.152973,-0.062395,0.155660,-0.044841,-0.033304,-0.103122,-0.476966,-0.228616
ENSMUSG00000002459_ProteinCoding_1,Rgs20,-0.058223,0.079762,0.030773,-0.059018,-0.015886,-0.079844,0.051330,-0.114898,0.036650,-0.122336,-0.046331,0.032270,-0.214984,-0.023708,0.035593,0.066026
ENSMUSG00000033793_ProteinCoding_1,Atp6v1h,-0.047068,-0.100044,0.158983,-0.099739,0.111228,0.008472,0.056761,-0.001394,-0.030393,-0.005540,0.071964,0.100648,0.021667,-0.035087,-0.047037,-0.132816


In [26]:
corr_df.sort_values("Pvalb", ascending=False)[['Gene', 'Pvalb']].head(n=20)

,Gene,Pvalb
ENSMUSG00000068739_ProteinCoding_1,Sars1,0.668200
ENSMUSG00000026797_ProteinCoding_1,Stxbp1,0.637784
ENSMUSG00000020436_ProteinCoding_1,Gabrg2,0.502908
ENSMUSG00000026825_ProteinCoding_1,Dnm1,0.478116
ENSMUSG00000059534_other_1,Uqcr10,0.471633
ENSMUSG00000041658_ProteinCoding_1,Rragb,0.457939
ENSMUSG00000060261_ProteinCoding_6,Gtf2i,0.437610
ENSMUSG00000026150_other_1,Mff,0.436462
ENSMUSG00000033530_other_1,Ttc7b,0.414459
ENSMUSG00000022378_ProteinCoding_2,Cyrib,0.384053


In [24]:
corr_df.sort_values("Sncg", ascending=False)[['Gene', 'Sncg']].head(n=20)

,Gene,Sncg
ENSMUSG00000017978_ProteinCoding_2,Cadps2,0.549435
ENSMUSG00000003500_ProteinCoding_1,Impdh1,0.408701
ENSMUSG00000054728_ProteinCoding_1,Phactr1,0.375240
ENSMUSG00000041658_ProteinCoding_1,Rragb,0.372348
ENSMUSG00000032076_ProteinCoding_3,Cadm1,0.372203
ENSMUSG00000039542_ProteinCoding_1,Ncam1,0.369225
ENSMUSG00000028863_NMD_3,Meaf6,0.356476
ENSMUSG00000027634_ProteinCoding_1,Ndrg3,0.350274
ENSMUSG00000078812_ProteinCoding_6,Eif5a,0.327042
ENSMUSG00000057738_ProteinCoding_5,Sptan1,0.318177


In [25]:
corr_df.to_csv(f"data/corrs/{data_source}_PSI_vs_exon_corr.csv")